# Introducción

## Título

Se utilizará la métrica sugerida:

Similitud = suma de longitudes de subcadenas comunes / longitud de programa más corto


# Configuración Global

## Importación de librerías

tokenize: Biblioteca estándar de Python para análisis léxico de código fuente de Python.

difflib: Biblioteca estándar de Python para comparación de secuencias y detección de diferencias.

In [6]:
import os
import re
import io
import keyword

import difflib
import tokenize

## Variables globales

DATASET_PATH = Ruta al directorio de archivos.py.

TARGET_FILE = Archivo que se evaluará contra todos los demás.

In [32]:
DATASET_PATH = "Python_Dataset/"
TARGET_FILE = "6.py"

# Funciones auxiliares

## Lectura del contenido de un archivo dado su Path



In [8]:
def read_file(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}")
    with open(path, 'r', encoding='utf-8') as f:
        return f.read()

## Tokenizar código

Convierte código fuente Python en una lista de tokens normalizados.
    
Ignora comentarios, saltos de línea, indentación y tokens de control.

Las cadenas de texto se reemplazan por 'STR' y los identificadores se convierten a minúsculas para reducir diferencias superficiales.
    

In [53]:
def tokenize_code(code):
    tokens = []
    try:
        reader = io.StringIO(code).readline
        for tok in tokenize.generate_tokens(reader):
            ttype = tok.type
            tval  = tok.string

            # Ignora tokens que no aportan información semántica
            if ttype in (tokenize.NEWLINE, tokenize.NL, tokenize.INDENT,
                         tokenize.DEDENT, tokenize.COMMENT,
                         tokenize.ENCODING, tokenize.ENDMARKER):
                continue

            if ttype == tokenize.NAME and keyword.iskeyword(tval):
                tokens.append(tval.lower())   # Palabra reservada
            elif ttype == tokenize.NAME:
                tokens.append(tval.lower())   # Identificador
            elif ttype == tokenize.NUMBER:
                tokens.append(tval)    
            elif ttype == tokenize.STRING:
                tokens.append(tval)    

    except tokenize.TokenError as e:
        print(f"Error al tokenizar: {e}")

    return tokens

## Comparación de texto llano

Usando usando libdiff compara dos códigos fuentes como texto plano línea por línea. 

Utiliza SequenceMatcher de difflib para encontrar bloques de lineas idénticas entre ambos archivos sin procesamiento previo.

In [52]:
def compare_plain_text(code_a, code_b):
    lines_a = code_a.splitlines()
    lines_b = code_b.splitlines()

    # SequenceMatcher encuentra la subsecuencia común más larga entre ambas listas
    matcher = difflib.SequenceMatcher(None, lines_a, lines_b)

    # Filtra bloques vacíos
    blocks = [
        (block.a, block.b, block.size)
        for block in matcher.get_matching_blocks()
        if block.size > 0
    ]

    # Métrica: suma de líneas coincidentes / líneas del programa más corto
    total_matching = sum(size for _, _, size in blocks)
    shorter = min(len(lines_a), len(lines_b))
    percentage = (total_matching / shorter) * 100 if shorter > 0 else 0

    return percentage, blocks

## Comparación de texto preprocesado

Usando libdiff y el código tokenizado Compara dos programas a nivel de tokens previamente normalizados.

A diferenciad la comparación por texto plano, este opera sobre la lista de tokens generada por el tokenizador, eliminando diferencias superficiales como espacios, comentarios o formato.

El coeficiente de similitud de Dice pero aplicado a tokens en lugar de líneas.

In [51]:
def compare_tokens(tokens_a, tokens_b):
    # SequenceMatcher encuentra la subsecuencia común más larga entre tokens
    matcher = difflib.SequenceMatcher(None, tokens_a, tokens_b)

    # Filtra bloques vacíos
    blocks = [
        (block.a, block.b, block.size)
        for block in matcher.get_matching_blocks()
        if block.size > 0
    ]

    # Métrica: suma de tokens coincidentes / tokens del programa más corto
    total_matching = sum(size for _, _, size in blocks)
    shorter = min(len(tokens_a), len(tokens_b))
    percentage = (total_matching / shorter) * 100 if shorter > 0 else 0
    
    return percentage, blocks

## Comparación mediante suffix array

build_suffix_array: Construye un suffix array apartrir de una lista de tokens, y retorna los índices de los sufijos ordenados lexicográficamente.

lcp_lenght: Calcula la longitud del prefijo común (Longest common Preffix) entre dos sufijos que inician en las posiciones i y j dentro de tokens.

compare_suffix_array: Concatena ambas listas con un separador único, construye el suffix array
del texto combinado y busca pares adyacentes que provengan de archivos
distintos para encontrar subcadenas comunes.
La similitud se calcula como la suma de longitudes de subcadenas comunes
dividida entre la longitud del programa más corto (métrica propuesta).


In [29]:
def build_suffix_array(tokens):
    """
    Construye un suffix array a partir de una lista de tokens.
    Retorna los índices de los sufijos ordenados lexicográficamente.
    """
    n = len(tokens)
    suffixes = [(tokens[i:], i) for i in range(n)]
    suffixes.sort(key=lambda x: x[0])
    return [s[1] for s in suffixes]


def lcp_length(tokens, i, j):
    """
    Calcula la longitud del prefijo común (LCP) entre dos sufijos
    que inician en las posiciones i y j dentro de tokens.
    """
    length = 0
    while i < len(tokens) and j < len(tokens) and tokens[i] == tokens[j]:
        length += 1
        i += 1
        j += 1
    return length


def compare_suffix_array(tokens_a, tokens_b, min_length=3):
    """
    Compara dos programas usando suffix array sobre tokens normalizados.
    Concatena ambas listas con un separador único, construye el suffix array
    del texto combinado y busca pares adyacentes que provengan de archivos
    distintos para encontrar subcadenas comunes.
    La similitud se calcula como tokens únicos cubiertos en A divididos
    entre la longitud del programa más corto.

    Parámetros:
        tokens_a (list[str]): Tokens del archivo objetivo.
        tokens_b (list[str]): Tokens del archivo a comparar.
        min_length (int): Longitud mínima de subcadena a considerar (default: 3).

    Retorna:
        tuple:
            - percentage (float): Porcentaje de similitud entre 0 y 100.
            - matches (list[tuple]): Lista de coincidencias (pos_a, pos_b, longitud).
    """
    # Combinar ambas listas con separador que no aparece en el código
    separator = ['$SEP$']
    combined = tokens_a + separator + tokens_b
    n_a = len(tokens_a)

    # Construir suffix array del texto combinado
    sa = build_suffix_array(combined)

    # Buscar pares adyacentes en el SA que pertenezcan a archivos distintos
    matches = []
    for idx in range(1, len(sa)):
        i = sa[idx - 1]
        j = sa[idx]

        i_in_a = i < n_a
        j_in_a = j < n_a

        # Solo comparar si uno está en A y el otro en B
        if i_in_a != j_in_a:
            lcp = lcp_length(combined, i, j)
            if lcp >= min_length and '$SEP$' not in combined[i:i+lcp]:
                matches.append((
                    i if i_in_a else j,     # posición en A
                    j if not j_in_a else i, # posición en B
                    lcp                     # longitud de la subcadena común
                ))

    # Deduplicar posiciones cubiertas en A para evitar solapamientos
    covered = set()
    for pos_a, _, length in matches:
        for k in range(length):
            covered.add(pos_a + k)

    # Métrica: tokens únicos cubiertos en A / tokens del programa más corto
    shorter = min(len(tokens_a), len(tokens_b))
    percentage = (len(covered) / shorter) * 100 if shorter > 0 else 0

    return percentage, matches

## Impresión

Imprime un resumen de similitud entre el archivo objetivo y un archivo comparado

In [50]:
def print_summary(filename_b, pct_plain, pct_tokens, pct_suffix):
    promedio = (pct_plain + pct_tokens + pct_suffix) / 3

    print("=" * 40)
    print(f"  Comparando contra: {filename_b}")
    print("=" * 40)
    print(f"  Texto plano:   {pct_plain:.1f}%")
    print(f"  Tokens:        {pct_tokens:.1f}%")
    print(f"  Suffix array:  {pct_suffix:.1f}%")
    print(f"  Promedio:      {promedio:.1f}%")
    print("=" * 40)

    if promedio >= 50:
        print("  Veredicto: MUY SIMILAR ")
    elif promedio >= 25:
        print("  Veredicto: MODERADAMENTE SIMILAR ")
    else:
        print("  Veredicto: POCO SIMILAR ")
    print("=" * 40)
    print()

# Comparaciones

## Ejecución principal

Toma el primer objetivo a comparar y lo evalúa contra todos los elementos del dataset

In [48]:
target_path = os.path.join(DATASET_PATH, TARGET_FILE)
code_target = read_file(target_path)
tokens_target = tokenize_code(code_target)

for filename in sorted(os.listdir(DATASET_PATH), key=lambda x: int(x.replace('.py', ''))):
    if filename.endswith(".py") and filename != TARGET_FILE:
        code_other = read_file(os.path.join(DATASET_PATH, filename))
        tokens_other = tokenize_code(code_other)

        pct_plain, blocks_plain = compare_plain_text(code_target, code_other)
        pct_tokens, blocks_tokens     = compare_tokens(tokens_target, tokens_other)
        pct_suffix, matches_suffix    = compare_suffix_array(tokens_target, tokens_other)

        print_summary(filename, pct_plain, pct_tokens, pct_suffix)

  Comparando contra: 1.py
  Texto plano:   1.4%
  Tokens:        2.3%
  Suffix array:  1.4%
  Promedio:      1.7%
  Veredicto: POCO SIMILAR 

  Comparando contra: 2.py
  Texto plano:   43.1%
  Tokens:        49.8%
  Suffix array:  87.7%
  Promedio:      60.2%
  Veredicto: MUY SIMILAR 

  Comparando contra: 3.py
  Texto plano:   0.0%
  Tokens:        55.1%
  Suffix array:  27.8%
  Promedio:      27.7%
  Veredicto: MODERADAMENTE SIMILAR 

  Comparando contra: 4.py
  Texto plano:   0.0%
  Tokens:        30.9%
  Suffix array:  2.1%
  Promedio:      11.0%
  Veredicto: POCO SIMILAR 

  Comparando contra: 5.py
  Texto plano:   47.2%
  Tokens:        49.8%
  Suffix array:  87.7%
  Promedio:      61.6%
  Veredicto: MUY SIMILAR 

  Comparando contra: 7.py
  Texto plano:   23.9%
  Tokens:        56.4%
  Suffix array:  27.6%
  Promedio:      36.0%
  Veredicto: MODERADAMENTE SIMILAR 

  Comparando contra: 8.py
  Texto plano:   0.0%
  Tokens:        2.9%
  Suffix array:  5.3%
  Promedio:      2.7%
 